In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [61]:
df = pd.read_csv('train.csv')
df.head()

,feedrate,clamp_pressure,tool_condition,machining_finalized,passed_visual_inspection,X1_delVal,X1_delAcc,X1_OutputPower,Y1_delVel,Y1_delAcc,...,Z1_ActualVelocity,Z1_ActualAcceleration,Z1_CommandVelocity,Z1_CommandAcceleration,S1_ActualVelocity,S1_ActualAcceleration,S1_CommandVelocity,S1_CommandAcceleration,S1_OutputPower.1,M1_CURRENT_FEEDRATE.1
0,6,4.0,unworn,yes,yes,0.001209,0.021075,0.246460,0.007059,0.092975,...,0.504599,0.514219,0.505388,0.501675,0.993194,0.436331,1.000000,8.280000e-09,0.400477,NaN
1,20,4.0,unworn,yes,yes,0.003438,0.015694,0.105820,0.015364,0.035067,...,0.499406,0.482434,0.499406,0.500000,0.563154,0.426633,0.553795,9.310000e-09,0.179883,0.168317
2,6,3.0,unworn,yes,yes,0.003438,0.015694,0.105820,0.015364,0.035067,...,0.499406,0.482434,0.499406,0.500000,0.563154,0.426633,0.553795,9.310000e-09,0.179883,0.168317
3,6,2.5,unworn,no,NaN,0.000099,0.631792,0.012468,0.000895,0.226735,...,0.403094,0.499949,0.402628,0.333333,0.384590,0.387422,0.385714,6.270000e-10,0.126425,0.380952
4,20,3.0,unworn,no,NaN,0.000450,0.025254,0.172509,0.001128,0.024071,...,0.344928,0.371797,0.343679,0.500000,0.221206,0.306763,0.219780,9.640000e-09,0.076350,0.668831


In [63]:
# Fill NaN values with the mode for categorical columns
categorical_cols = ['tool_condition', 'machining_finalized', 'passed_visual_inspection']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])


In [64]:
# Fill NaN values for continuous columns with mean or median
continuous_cols = ['feedrate', 'clamp_pressure', 'X1_delVal', 'X1_delAcc', 'X1_OutputPower',
                   'Y1_delVel', 'Y1_delAcc', 'Y1_OutputPower', 'Z1_delVel', 'Z1_delAcc',
                   'S1_delVel', 'S1_delAcc', 'S1_OutputPower', 'M1_CURRENT_FEEDRATE']
for col in continuous_cols:
    df[col] = df[col].fillna(df[col].mean())  # Or use df[col].median()


In [67]:
# Extracting X and Y
X = df.iloc[:, [0, 1] + list(range(5, 17))]  # Input features: columns 0, 1, 5 to 16
Y = df.iloc[:, [2, 3, 4]]  # Output features: columns 2, 3, 4


In [70]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Normalize the data (optional but usually helps with training)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
le_tool_condition = LabelEncoder()
le_machining_finalized = LabelEncoder()
le_visual_inspection = LabelEncoder()

# Fit and transform the categorical variables
y_train['tool_condition'] = le_tool_condition.fit_transform(y_train['tool_condition'])
y_train['machining_finalized'] = le_machining_finalized.fit_transform(y_train['machining_finalized'])
y_train['passed_visual_inspection'] = le_visual_inspection.fit_transform(y_train['passed_visual_inspection'])

y_test['tool_condition'] = le_tool_condition.transform(y_test['tool_condition'])
y_test['machining_finalized'] = le_machining_finalized.transform(y_test['machining_finalized'])
y_test['passed_visual_inspection'] = le_visual_inspection.transform(y_test['passed_visual_inspection'])




In [74]:
from keras.models import Model
from keras.layers import Input, Dense
from keras.optimizers import Adam

# Define the input layer
inputs = Input(shape=(X_train.shape[1],))

# Define hidden layers
x = Dense(64, activation='relu')(inputs)
x = Dense(32, activation='relu')(x)

# Define output layers
output_tool_condition = Dense(1, activation='sigmoid', name='tool_condition')(x)
output_machining_finalized = Dense(1, activation='sigmoid', name='machining_finalized')(x)
output_visual_inspection = Dense(1, activation='sigmoid', name='passed_visual_inspection')(x)

# Define the model with multiple outputs
model = Model(inputs=inputs, outputs=[output_tool_condition, output_machining_finalized, output_visual_inspection])

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics={
        'tool_condition': ['accuracy'],
        'machining_finalized': ['accuracy'],
        'passed_visual_inspection': ['accuracy']
    }
)

# Train the model
model.fit(X_train, {
    'tool_condition': y_train['tool_condition'],
    'machining_finalized': y_train['machining_finalized'],
    'passed_visual_inspection': y_train['passed_visual_inspection']
}, epochs=10, batch_size=32)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 1.8971 - machining_finalized_accuracy: 0.6429 - machining_finalized_loss: 0.6333 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.5093 - tool_condition_accuracy: 0.2857 - tool_condition_loss: 0.7546
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - loss: 1.8499 - machining_finalized_accuracy: 0.6429 - machining_finalized_loss: 0.6132 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.4931 - tool_condition_accuracy: 0.3571 - tool_condition_loss: 0.7436
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 1.8044 - machining_finalized_accuracy: 0.7143 - machining_finalized_loss: 0.5938 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.4777 - tool_condition_accuracy: 0.3571 - tool_condition_loss: 0.7330
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 1.7616 - machining_finalized_accuracy: 0.7143 - machining_finalized_loss: 0.5749 - pa

Hyper Parameter Tuning


In [76]:
pip install keras-tuner


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 9.0 MB/s eta 0:00:00


In [79]:
import keras_tuner as kt
from keras.models import Model
from keras.layers import Input, Dense
from keras.optimizers import Adam

# Define the model creation function
def build_model(hp):
    inputs = Input(shape=(X_train.shape[1],))

    # Hyperparameters for number of units in hidden layers
    x = Dense(units=hp.Int('units', min_value=32, max_value=128, step=32), activation='relu')(inputs)
    x = Dense(units=hp.Int('units', min_value=32, max_value=128, step=32), activation='relu')(x)

    # Output layers for each target variable
    output_tool_condition = Dense(1, activation='sigmoid', name='tool_condition')(x)
    output_machining_finalized = Dense(1, activation='sigmoid', name='machining_finalized')(x)
    output_visual_inspection = Dense(1, activation='sigmoid', name='passed_visual_inspection')(x)

    # Define the model
    model = Model(inputs=inputs, outputs=[output_tool_condition, output_machining_finalized, output_visual_inspection])

    # Compile the model with hyperparameters
    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-5, max_value=1e-2, sampling='LOG')),
        loss='binary_crossentropy',
        metrics ={
        'tool_condition': ['accuracy', 'AUC'],
        'machining_finalized': ['accuracy', 'AUC'],
        'passed_visual_inspection': ['accuracy', 'AUC']
        }
    )

    return model

# Define the tuner (using Random Search in this example)
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',  # Search based on validation loss
    max_trials=10,  # Number of different hyperparameter combinations to try
    executions_per_trial=1,  # How many times each combination is trained
    directory='my_dir',  # Directory to store the results
    project_name='hyperparameter_tuning'
)

# Start the search for the best hyperparameters
tuner.search(X_train, {
    'tool_condition': y_train['tool_condition'],
    'machining_finalized': y_train['machining_finalized'],
    'passed_visual_inspection': y_train['passed_visual_inspection']
}, epochs=10, batch_size=32, validation_data=(X_test, {
    'tool_condition': y_test['tool_condition'],
    'machining_finalized': y_test['machining_finalized'],
    'passed_visual_inspection': y_test['passed_visual_inspection']
}))

# Get the best model and hyperparameters
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

# Print the best hyperparameters
print(f"Best hyperparameters: {best_hyperparameters.values}")


Trial 10 Complete [00h 00m 08s]
val_loss: 1.7256736755371094

Best val_loss So Far: 1.693599820137024
Total elapsed time: 00h 04m 43s
Best hyperparameters: {'units': 128, 'learning_rate': 0.009828563132511563}


Fully connected (dense) feedforward neural network

In [82]:
from keras.models import Model
from keras.layers import Input, Dense
from keras.optimizers import Adam

# Define the input layer
inputs = Input(shape=(X_train.shape[1],))

# Define hidden layers
x = Dense(64, activation='relu')(inputs)
x = Dense(32, activation='relu')(x)

# Define output layers
output_tool_condition = Dense(1, activation='sigmoid', name='tool_condition')(x)
output_machining_finalized = Dense(1, activation='sigmoid', name='machining_finalized')(x)
output_visual_inspection = Dense(1, activation='sigmoid', name='passed_visual_inspection')(x)

# Define the model with multiple outputs
model = Model(inputs=inputs, units = 128, learning_rate = 0.009828563132511563, outputs=[output_tool_condition, output_machining_finalized, output_visual_inspection])

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics={
        'tool_condition': ['accuracy'],
        'machining_finalized': ['accuracy'],
        'passed_visual_inspection': ['accuracy']
    }
)

# Train the model
model.fit(X_train, {
    'tool_condition': y_train['tool_condition'],
    'machining_finalized': y_train['machining_finalized'],
    'passed_visual_inspection': y_train['passed_visual_inspection']
}, epochs=10, batch_size=32)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - loss: 2.0020 - machining_finalized_accuracy: 0.5000 - machining_finalized_loss: 0.6832 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.5370 - tool_condition_accuracy: 0.5714 - tool_condition_loss: 0.7819
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - loss: 1.9514 - machining_finalized_accuracy: 0.5000 - machining_finalized_loss: 0.6538 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.5249 - tool_condition_accuracy: 0.5714 - tool_condition_loss: 0.7727
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - loss: 1.9031 - machining_finalized_accuracy: 0.6429 - machining_finalized_loss: 0.6262 - passed_visual_inspection_accuracy: 0.8571 - passed_visual_inspection_loss: 0.5129 - tool_condition_accuracy: 0.5714 - tool_condition_loss: 0.7641
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - loss: 1.8567 - machining_finalized_accuracy: 0.8571 - machining_finalized_loss: 0.6001 - p